In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.9 MB/s eta 0:00:00:00:010:01m


In [3]:
"""
Visual Product Search Engine - Batch Evaluation Script
Evaluates Config A, B (α=0.7, 0.5), and C (α=0.7, 0.5) on full query set.
Computes: Recall@K, NDCG@K, mAP@K for K ∈ {5, 10, 15}
Reports: mean ± std over multiple random seeds
"""

import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION - Kaggle paths
# ============================================================================

CROPS_DATASET      = '/kaggle/input/datasets/harshitabansal307/yolo-bbox-crops-v1'
INDEXES_AB_DATASET = '/kaggle/input/datasets/likithareddy2508/clip-indexes-ab'
INDEXES_C_DATASET  = '/kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c'

# Random seeds (team roll numbers)
SEEDS = [550, 537, 585, 35]

# Evaluation parameters
K_VALUES        = [5, 10, 15]
BATCH_SIZE      = 64
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'

# ============================================================================
# SETUP
# ============================================================================

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device   : {DEVICE}')
print(f'Seeds    : {SEEDS}')
print(f'K values : {K_VALUES}')
print()

# Check paths
for label, path in [
    ('CROPS_DATASET',      CROPS_DATASET),
    ('INDEXES_AB_DATASET', INDEXES_AB_DATASET),
    ('INDEXES_C_DATASET',  INDEXES_C_DATASET),
]:
    status = 'Found ✓' if os.path.exists(path) else 'NOT FOUND ✗'
    print(f'[{status}] {label}: {path}')
print()

# ============================================================================
# LOAD DATA
# ============================================================================

print('Loading data...')

master_df = pd.read_csv(os.path.join(CROPS_DATASET, 'master_crops.csv'))
query_df  = master_df[master_df['split'] == 'query'].reset_index(drop=True)

def remap_path(old_path):
    if pd.isna(old_path):
        return old_path
    relative = old_path.replace('/kaggle/working/', '').replace('/kaggle/input/', '')
    for ds in ['yolocroppeddataset/', 'yolo-bbox-crops-v1/',
               'datasets/harshitabansal307/yolo-bbox-crops-v1/']:
        relative = relative.replace(ds, '')
    return os.path.join(CROPS_DATASET, relative)

query_df['crop_path']   = query_df['crop_path'].apply(remap_path)
query_df['crop_exists'] = query_df['crop_path'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

# Fallback path construction if remapping did not work
if query_df['crop_exists'].sum() < len(query_df) * 0.9:
    print('Primary path remapping insufficient — trying direct construction...')
    def direct_path(image_name):
        relative = image_name[4:] if image_name.startswith('img/') else image_name
        for subdir in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(CROPS_DATASET, subdir, relative)
            if os.path.exists(p):
                return p
        return os.path.join(CROPS_DATASET, 'data/bbox_crops', relative)
    query_df['crop_path']   = query_df['image_name'].apply(direct_path)
    query_df['crop_exists'] = query_df['crop_path'].apply(os.path.exists)

valid_query_df = query_df[query_df['crop_exists']].reset_index(drop=True)
print(f'Query images found : {len(valid_query_df):,} / {len(query_df):,}')

# Load gallery metadata
gallery_meta        = pd.read_csv(os.path.join(INDEXES_AB_DATASET, 'gallery_metadata.csv'))
gallery_item_ids    = gallery_meta['item_id'].tolist()
gallery_item_counts = gallery_meta['item_id'].value_counts().to_dict()

print(f'Gallery items      : {len(gallery_meta):,}')
print(f'Unique item_ids    : {gallery_meta["item_id"].nunique():,}')
print()

# ============================================================================
# LOAD CLIP MODEL (frozen)
# ============================================================================

print('Loading CLIP model...')
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
for param in clip_model.parameters():
    param.requires_grad = False
clip_model.eval()
EMBED_DIM = clip_model.config.projection_dim
print(f'CLIP loaded! Embedding dim: {EMBED_DIM}')
print()

# ============================================================================
# ENCODE ALL QUERY IMAGES
# ============================================================================

def encode_query_images(model):
    n    = len(valid_query_df)
    embs = np.zeros((n, EMBED_DIM), dtype=np.float32)

    for batch_start in tqdm(range(0, n, BATCH_SIZE), desc='Encoding queries'):
        batch = valid_query_df.iloc[batch_start : batch_start + BATCH_SIZE]
        images, valid_idx = [], []

        for i, (_, row) in enumerate(batch.iterrows()):
            try:
                images.append(Image.open(row['crop_path']).convert('RGB'))
                valid_idx.append(i)
            except Exception:
                pass

        if not images:
            continue

        inputs = clip_processor(images=images, return_tensors='pt', padding=True).to(DEVICE)
        with torch.no_grad():
            out   = model.get_image_features(**inputs)
            feats = out.pooler_output if (hasattr(out, 'pooler_output')
                                         and not isinstance(out, torch.Tensor)) else out
        feats = feats / feats.norm(dim=-1, keepdim=True)

        for local_i, global_i in enumerate(valid_idx):
            embs[batch_start + global_i] = feats[local_i].cpu().numpy()

    return embs


print('Encoding query images with frozen CLIP (Configs A & B)...')
query_embs_frozen = encode_query_images(clip_model)
print(f'Shape: {query_embs_frozen.shape}')
print()

# ============================================================================
# ENCODE QUERY IMAGES WITH FINE-TUNED CLIP (Config C)
# ============================================================================

ft_weights_path = os.path.join(INDEXES_C_DATASET, 'clip_finetuned_full.pt')

if os.path.exists(ft_weights_path):
    print('Loading fine-tuned CLIP weights for Config C...')
    state_dict = torch.load(ft_weights_path, map_location=DEVICE)
    clip_model.load_state_dict(state_dict)
    clip_model.eval()
    print('Fine-tuned weights loaded ✓')

    print('Encoding query images with fine-tuned CLIP (Config C)...')
    query_embs_finetuned = encode_query_images(clip_model)
    print(f'Shape: {query_embs_finetuned.shape}')
    print()
else:
    print(f'WARNING: Fine-tuned weights not found at: {ft_weights_path}')
    print('Config C will be SKIPPED.')
    query_embs_finetuned = None
    print()

# ============================================================================
# METRIC FUNCTIONS
# ============================================================================

def compute_ndcg_at_k(retrieved_matches, n_relevant, k):
    dcg  = sum(1.0 / np.log2(rank + 2)
               for rank, hit in enumerate(retrieved_matches[:k]) if hit)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(n_relevant, k)))
    return dcg / idcg if idcg > 0 else 0.0

def compute_map_at_k(retrieved_matches, n_relevant, k):
    hits, prec_sum = 0, 0.0
    for rank, hit in enumerate(retrieved_matches[:k], start=1):
        if hit:
            hits += 1
            prec_sum += hits / rank
    denom = min(n_relevant, k)
    return prec_sum / denom if denom > 0 else 0.0

# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_config(index_path, config_name, query_embs, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Sample 20% of queries (min 500)
    n_sample   = min(len(valid_query_df), max(500, int(0.2 * len(valid_query_df))))
    sample_idx = np.random.choice(len(valid_query_df), n_sample, replace=False)

    sample_embs     = query_embs[sample_idx].astype(np.float32)
    sample_item_ids = valid_query_df.iloc[sample_idx]['item_id'].tolist()

    # Batch search
    index = faiss.read_index(index_path)
    max_k = max(K_VALUES)
    _, indices = index.search(sample_embs, max_k)

    per_k = {k: {'recall': [], 'ndcg': [], 'map': []} for k in K_VALUES}

    for q_item_id, result_idx in zip(sample_item_ids, indices):
        retrieved_ids     = [gallery_item_ids[i] for i in result_idx
                             if i < len(gallery_item_ids)]
        retrieved_matches = [1 if r == q_item_id else 0 for r in retrieved_ids]
        n_relevant        = gallery_item_counts.get(q_item_id, 1)

        for k in K_VALUES:
            per_k[k]['recall'].append(1 if any(retrieved_matches[:k]) else 0)
            per_k[k]['ndcg'].append(compute_ndcg_at_k(retrieved_matches, n_relevant, k))
            per_k[k]['map'].append(compute_map_at_k(retrieved_matches, n_relevant, k))

    result_dict = {}
    for k in K_VALUES:
        result_dict[f'recall@{k}'] = np.mean(per_k[k]['recall'])
        result_dict[f'ndcg@{k}']   = np.mean(per_k[k]['ndcg'])
        result_dict[f'map@{k}']    = np.mean(per_k[k]['map'])

    return result_dict

# ============================================================================
# DEFINE ALL CONFIGS
# ============================================================================

CONFIGS = []

def try_add_config(name, filename, dataset, query_embs):
    p = os.path.join(dataset, filename)
    if os.path.exists(p):
        CONFIGS.append((name, p, query_embs))
        print(f'  Added: {name}')
    else:
        print(f'  WARNING: Skipping {name} — not found: {p}')

print('Checking config indexes...')
try_add_config('Config_A_alpha1.0',  'faiss_index_A_alpha10.bin', INDEXES_AB_DATASET, query_embs_frozen)
try_add_config('Config_B_alpha0.7',  'faiss_index_B_alpha07.bin', INDEXES_AB_DATASET, query_embs_frozen)
try_add_config('Config_B_alpha0.5',  'faiss_index_B_alpha05.bin', INDEXES_AB_DATASET, query_embs_frozen)

if query_embs_finetuned is not None:
    try_add_config('Config_C_alpha0.7', 'faiss_index_C_alpha07.bin', INDEXES_C_DATASET, query_embs_finetuned)
    try_add_config('Config_C_alpha0.5', 'faiss_index_C_alpha05.bin', INDEXES_C_DATASET, query_embs_finetuned)

print()

# ============================================================================
# RUN EVALUATION
# ============================================================================

print('Starting evaluation...')
print()

all_results = {}

for config_name, index_path, query_embs in CONFIGS:
    print(f'=== {config_name} ===')
    seed_results = []

    for seed in SEEDS:
        result = evaluate_config(index_path, config_name, query_embs, seed)
        seed_results.append(result)
        print(f'  Seed {seed:>4} | '
              f'Recall@10={result["recall@10"]:.4f}  '
              f'NDCG@10={result["ndcg@10"]:.4f}  '
              f'mAP@10={result["map@10"]:.4f}')

    seed_df = pd.DataFrame(seed_results)
    all_results[config_name] = {'mean': seed_df.mean(), 'std': seed_df.std()}

    print(f'  MEAN  | '
          f'Recall@10={seed_df["recall@10"].mean():.4f} ± {seed_df["recall@10"].std():.4f}')
    print()

# ============================================================================
# PRINT FINAL RESULTS TABLE
# ============================================================================

print('=' * 80)
print('  FINAL RESULTS — Visual Product Search Engine')
print(f'  Seeds used: {SEEDS}   |   Format: mean ± std')
print('=' * 80)
print()

for config_name, _, _ in CONFIGS:
    if config_name not in all_results:
        continue
    mean = all_results[config_name]['mean']
    std  = all_results[config_name]['std']

    print(f'Config: {config_name}')
    print(f'  {"K":>4}  {"Recall@K":>18}  {"NDCG@K":>18}  {"mAP@K":>18}')
    print(f'  {"─" * 62}')
    for k in K_VALUES:
        print(
            f'  K={k:>2}  '
            f'{mean[f"recall@{k}"]:.4f} ± {std[f"recall@{k}"]:.4f}  '
            f'{mean[f"ndcg@{k}"]:.4f} ± {std[f"ndcg@{k}"]:.4f}  '
            f'{mean[f"map@{k}"]:.4f} ± {std[f"map@{k}"]:.4f}'
        )
    print()

# ============================================================================
# SAVE TO CSV
# ============================================================================

rows = []
for config_name, _, _ in CONFIGS:
    if config_name not in all_results:
        continue
    mean = all_results[config_name]['mean']
    std  = all_results[config_name]['std']
    for k in K_VALUES:
        rows.append({
            'Config':      config_name,
            'K':           k,
            'Recall_mean': round(mean[f'recall@{k}'], 4),
            'Recall_std':  round(std[f'recall@{k}'],  4),
            'NDCG_mean':   round(mean[f'ndcg@{k}'],   4),
            'NDCG_std':    round(std[f'ndcg@{k}'],    4),
            'mAP_mean':    round(mean[f'map@{k}'],    4),
            'mAP_std':     round(std[f'map@{k}'],     4),
        })

results_df = pd.DataFrame(rows)
out_path   = '/kaggle/working/evaluation_results_final.csv'
results_df.to_csv(out_path, index=False)
print(f'Results saved to {out_path}')

# ============================================================================
# BEST CONFIG SUMMARY
# ============================================================================

valid_configs = [c[0] for c in CONFIGS if c[0] in all_results]
if valid_configs:
    best_config = max(valid_configs,
                      key=lambda c: all_results[c]['mean']['recall@10'])
    best_val    = all_results[best_config]['mean']['recall@10']
    print()
    print('=' * 80)
    print(f'BEST CONFIG (by Recall@10): {best_config}')
    print(f'Recall@10 = {best_val:.4f}')
    print('Use this config in your Streamlit demo!')
    print('=' * 80)


Device   : cuda
Seeds    : [550, 537, 585, 35]
K values : [5, 10, 15]

[Found ✓] CROPS_DATASET: /kaggle/input/datasets/harshitabansal307/yolo-bbox-crops-v1
[Found ✓] INDEXES_AB_DATASET: /kaggle/input/datasets/likithareddy2508/clip-indexes-ab
[Found ✓] INDEXES_C_DATASET: /kaggle/input/datasets/likithareddy2508/clip-finetuned-indexes-c

Loading data...
Query images found : 14,218 / 14,218
Gallery items      : 12,612
Unique item_ids    : 3,985

Loading CLIP model...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

CLIP loaded! Embedding dim: 512

Encoding query images with frozen CLIP (Configs A & B)...


Encoding queries: 100%|██████████| 223/223 [01:57<00:00,  1.90it/s]


Shape: (14218, 512)

Loading fine-tuned CLIP weights for Config C...
Fine-tuned weights loaded ✓
Encoding query images with fine-tuned CLIP (Config C)...


Encoding queries: 100%|██████████| 223/223 [01:18<00:00,  2.86it/s]


Shape: (14218, 512)

Checking config indexes...
  Added: Config_A_alpha1.0
  Added: Config_B_alpha0.7
  Added: Config_B_alpha0.5
  Added: Config_C_alpha0.7
  Added: Config_C_alpha0.5

Starting evaluation...

=== Config_A_alpha1.0 ===
  Seed  550 | Recall@10=0.5652  NDCG@10=0.2420  mAP@10=0.1700
  Seed  537 | Recall@10=0.5519  NDCG@10=0.2274  mAP@10=0.1558
  Seed  585 | Recall@10=0.5853  NDCG@10=0.2408  mAP@10=0.1642
  Seed   35 | Recall@10=0.5558  NDCG@10=0.2384  mAP@10=0.1672
  MEAN  | Recall@10=0.5645 ± 0.0149

=== Config_B_alpha0.7 ===
  Seed  550 | Recall@10=0.5596  NDCG@10=0.2396  mAP@10=0.1696
  Seed  537 | Recall@10=0.5480  NDCG@10=0.2256  mAP@10=0.1554
  Seed  585 | Recall@10=0.5779  NDCG@10=0.2396  mAP@10=0.1643
  Seed   35 | Recall@10=0.5667  NDCG@10=0.2382  mAP@10=0.1664
  MEAN  | Recall@10=0.5630 ± 0.0125

=== Config_B_alpha0.5 ===
  Seed  550 | Recall@10=0.5192  NDCG@10=0.2108  mAP@10=0.1462
  Seed  537 | Recall@10=0.5100  NDCG@10=0.1983  mAP@10=0.1338
  Seed  585 | Recall